In [2]:
# 1. 파이썬 코드에서 Matplotlib 폰트 설정
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from datetime import timedelta
from itertools import combinations
from collections import Counter

# 폰트 설정
plt.rc('font', family='Malgun Gothic')
# 마이너스 부호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정이 완료되었습니다.")

# 3. 데이터 로드 및 통합 (모든 문제 풀이의 시작점)
try:
    orders_df = pd.read_csv('../data/orders.csv')
    payments_df = pd.read_csv('../data/payments.csv')
    products_df = pd.read_csv('../data/products.csv')
    shipping_df = pd.read_csv('../data/shipping.csv')
    customers_df = pd.read_csv('../data/customers.csv')

    # 모든 데이터프레임 병합
    df = pd.merge(orders_df, payments_df, on='order_id', how='left')
    df = pd.merge(df, products_df, on='product_id', how='left')
    df = pd.merge(df, customers_df, on='customer_id', how='left')
    df = pd.merge(df, shipping_df, on='order_id', how='left')

    # 데이터 전처리
    date_cols = ['order_date', 'payment_date', 'join_date', 'shipping_start_date', 'shipping_end_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce') # pandas의 to_datetime 검색해서 사용법 잘 봐 두고
    df['total_sales'] = df['quantity'] * df['price']
    
    print("데이터 로드 및 통합이 완료되었습니다.")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다: {e}")


한글 폰트 설정이 완료되었습니다.
데이터 로드 및 통합이 완료되었습니다.


In [8]:
# 문제 3: 이탈 고객(가정: 마지막 구매 후 180일 이상 경과)의 이탈 직전 30일간 구매 패턴을 분석하세요.
# - 분석 기준일을 마지막 주문일 다음 날로 설정 (180일은 코딩하는 날 기준으로)
# 비즈니스 목적: 고객이 이탈하기 전에 보이는 공통적인 행동 패턴을 파악하여, 유사한 패턴을 보이는 고객에게 선제적으로 이탈 방지 캠페인을 진행합니다.

In [9]:
# 출력 결과를 보고 코딩하세요
resutls = '''
이탈 고객의 이탈 직전 30일 구매 패턴:
             avg_sales top_category
customer_id                        
C0001          23800.0        스포츠용품
C0006          68000.0           도서
C0009         194400.0           식품
C0013         121600.0        스포츠용품
C0014          94200.0           식품
'''

In [10]:
df.columns

Index(['order_id', 'customer_id', 'product_id', 'order_date', 'quantity',
       'payment_id', 'payment_method', 'payment_status', 'payment_date',
       'product_name', 'category', 'price', 'stock', 'name', 'gender', 'age',
       'join_date', 'city', 'shipping_id', 'shipping_company',
       'shipping_status', 'shipping_start_date', 'shipping_end_date',
       'total_sales'],
      dtype='object')

In [ ]:
# 필요한 컬럼 : 'customer_id', 'product_id', 'order_date', 'category',  'total_sales'
cols = ['customer_id', 'product_id', 'order_date', 'category', 'total_sales']
df_quest23 = df[cols].copy()
#df.query("customer_id == 'C0010'")


In [ ]:
# 기준일: 오늘 날짜, 오늘 일자 기준 180일 이전을 cutoff_date로 지정
today = pd.Timestamp.today()
cutoff_date = today - timedelta(days=180)

# 1. 이탈 고객 추출: 마지막 구매일이 180일 이상 지난 고객아이디와 최종주문일 찾기
target_customers = (
    df_quest23.groupby('customer_id') # 고객아이디별로 그룹핑
    ['order_date'].max() # 고객별 최종 주문일자 찾아서
    .reset_index()       # DF 로 만들기 위해 reset_index()하고
    .query("order_date <= @cutoff_date") # 최종 주문일자가 cutoff_date보다 작거나 같은 것만 찾아서
    .rename(columns={'order_date': 'last_order_date'}) # 주문일자를 최종주문일자로 컬럼명 변경
)
# target_customers.head()
# target_customers[target_customers['customer_id'] == 'C0010']                

In [ ]:
# 2. 이탈 고객의 전체 구매 이력를 고객아이디로 병합
df_last = pd.merge(df_quest23, target_customers, on='customer_id')
df_last.head()

,customer_id,product_id,order_date,category,total_sales,last_order_date
0,C0160,P0428,2023-10-20 11:14:49,스포츠용품,44000,2025-05-04 11:20:36
1,C1581,P0141,2023-10-20 13:55:14,스포츠용품,17900,2024-07-13 09:22:15
2,C1580,P0318,2023-10-20 18:23:41,의류,71500,2025-02-09 12:02:49
3,C0554,P0048,2023-10-20 21:44:58,화장품,77400,2024-07-27 17:50:54
4,C1310,P0324,2023-10-21 01:43:16,전자제품,283200,2024-10-19 08:01:40


In [16]:
# 3. 마지막 구매일 기준으로 직전 30일 데이터 필터링
df_last_month = df_last[
    df_last['order_date'].between(
        df_last['last_order_date'] - timedelta(days=30),
        df_last['last_order_date']
    )
]

# 4. 고객별 카테고리별 평균 구매액 계산
category_avg = (
    df_last_month.groupby(['customer_id', 'category'])['total_sales']
    .mean()
    .reset_index()
)

# 5. 고객별로 avg_sales가 가장 높은 카테고리 추출
top_category = (
    category_avg.sort_values(['customer_id', 'total_sales'], ascending=[True, False])
    .groupby('customer_id')
    .first()
    .reset_index()
)

# 6. 컬럼명 정리 및 최종 결과 출력
result = top_category.rename(columns={'total_sales': 'avg_sales', 'category': 'top_category'})
print('이탈 고객의 이탈 직전 30일 구매 패턴 : ')
print(result.set_index('customer_id'))

이탈 고객의 이탈 직전 30일 구매 패턴 : 
            top_category  avg_sales
customer_id                        
C0001              스포츠용품    23800.0
C0006                 도서    68000.0
C0009                 식품   194400.0
C0010                 식품   113700.0
C0013              스포츠용품   121600.0
...                  ...        ...
C1974                 식품   350400.0
C1975                 도서    16500.0
C1976                화장품    14000.0
C1989                화장품   265600.0
C1998                화장품    41200.0

[610 rows x 2 columns]
